# Can we predict whether a student passes the closing quiz (`passed_quiz`)?

Plan (follows `checklist.md`):
- **Split**: there is no row-level time column and `student_key` repeats (1,800 students, ~7 rows each), so the test set is a held-out set of *whole students*. Cross-validation inside the training rows uses `GroupKFold` on students.
- **Features**: `student_key` is an identifier and is dropped. `enrol_year`, `school_code`, `prior_score`, `session_minutes`, `hints_used` are recorded before the closing quiz, so they are allowed. Caveat: `session_minutes` and `hints_used` are only known once the session is nearly over, so this is a *pre-quiz* predictor, not a start-of-session one.
- Candidates are named before any scoring, preprocessing lives inside the CV pipelines, and the test set is scored exactly once.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, brier_score_loss

df = pd.read_csv('data.csv')
print(df.shape, int(df.isna().sum().sum()), 'missing;', df.student_key.nunique(), 'students')
print('pass rate', round(df.passed_quiz.mean(), 3))
df.groupby('student_key')[['enrol_year', 'school_code']].nunique().max()

## 1. Set aside the test set (whole students) before any modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert set(train.student_key).isdisjoint(test.student_key)
print(len(train), len(test), round(train.passed_quiz.mean(), 3), round(test.passed_quiz.mean(), 3))
TARGET = 'passed_quiz'; ID = 'student_key'
NUM = ['prior_score', 'session_minutes', 'hints_used', 'enrol_year']
CAT = ['school_code']
y_tr, g_tr = train[TARGET].values, train[ID].values
X_tr, X_te, y_te = train.drop(columns=[TARGET, ID]), test.drop(columns=[TARGET, ID]), test[TARGET].values

## 2. Candidate models (declared before scoring anything)
1. `baseline_prior`: predicts the training pass rate for every row (trivial).
2. `logreg_session`: logistic regression on `prior_score, session_minutes, hints_used`.
3. `logreg_all`: logistic regression on those plus `enrol_year` and one-hot `school_code`.
4. `hgb_all`: histogram gradient boosting on all five features (`max_depth=3`, `learning_rate=0.05`, `max_iter=200`).

Selection metric: out-of-fold ROC AUC. The decision threshold is then chosen on the winner's out-of-fold predictions by maximising balanced accuracy. No other hyperparameters are tuned.

In [ ]:
def lr(cols_num, cols_cat):
    tf = [('n', StandardScaler(), cols_num)]
    if cols_cat:
        tf.append(('c', OneHotEncoder(handle_unknown='ignore'), cols_cat))
    return Pipeline([('pre', ColumnTransformer(tf)), ('m', LogisticRegression(max_iter=1000))])

CANDIDATES = {
    'baseline_prior': lambda: Pipeline([('m', DummyClassifier(strategy='prior'))]),
    'logreg_session': lambda: lr(['prior_score', 'session_minutes', 'hints_used'], []),
    'logreg_all': lambda: lr(NUM, CAT),
    'hgb_all': lambda: Pipeline([
        ('pre', ColumnTransformer([('n', 'passthrough', NUM),
                                   ('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT)])),
        ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0))]),
}
list(CANDIDATES)

## 3. Grouped cross-validation on training rows only (preprocessing is inside each pipeline, so it is refit per fold)

In [ ]:
folds = list(GroupKFold(n_splits=5).split(X_tr, y_tr, groups=g_tr))
oof = {}; rows = []
for name, mk in CANDIDATES.items():
    p = np.zeros(len(train)); aucs = []
    for a, b in folds:
        m = mk().fit(X_tr.iloc[a], y_tr[a])
        p[b] = m.predict_proba(X_tr.iloc[b])[:, 1]
        aucs.append(roc_auc_score(y_tr[b], p[b]))
    oof[name] = p
    rows.append(dict(model=name, cv_auc_mean=np.mean(aucs), cv_auc_sd=np.std(aucs), cv_brier=brier_score_loss(y_tr, p)))
cv_res = pd.DataFrame(rows).set_index('model').round(4)
cv_res

## 4. Choose model and threshold on CV only

In [ ]:
best = cv_res.drop(index='baseline_prior').cv_auc_mean.idxmax()
ths = np.linspace(0.2, 0.7, 51)
bal = [balanced_accuracy_score(y_tr, oof[best] >= t) for t in ths]
THRESH = float(ths[int(np.argmax(bal))])
print('chosen model:', best, '| threshold:', round(THRESH, 2))

**Choice recorded before test scoring:** the model and threshold printed above are final (highest mean CV AUC among the non-trivial candidates; the CV table shows how far it is ahead of the baseline and of the other candidates).

## 5. Refit on all training rows; score the test set once (headline)

In [ ]:
final = CANDIDATES[best]().fit(X_tr, y_tr)
p_te = final.predict_proba(X_te)[:, 1]
pred = p_te >= THRESH
rng = np.random.default_rng(0)
sids = test.student_key.values; uniq = np.unique(sids)
idx_by = {s: np.where(sids == s)[0] for s in uniq}
bs = []
for _ in range(500):
    ii = np.concatenate([idx_by[s] for s in rng.choice(uniq, len(uniq))])
    bs.append(roc_auc_score(y_te[ii], p_te[ii]))
print(f'TEST ROC AUC = {roc_auc_score(y_te, p_te):.3f}  (95% student-bootstrap CI {np.percentile(bs, 2.5):.3f}-{np.percentile(bs, 97.5):.3f})')
print(f'balanced acc = {balanced_accuracy_score(y_te, pred):.3f}, accuracy = {accuracy_score(y_te, pred):.3f}, majority-class accuracy = {max(y_te.mean(), 1 - y_te.mean()):.3f}')

## 6. Interpretation
The headline is the single test AUC with its interval; other candidates were not scored on the test set. An AUC near 0.5 means the pre-quiz features carry little signal. Limitations: no time column, so this measures generalisation to new students, not to future sessions; `prior_score` varies across a student's rows despite being a placement score (possible recording noise); `session_minutes` and `hints_used` are only available late in the session.